# Student Performance Analysis

## Project Overview

This project explores the Student Performance Factors dataset to investigate how study habits, attendance, motivation, family support and other factors relate to exam scores.

The aim is to understand both individual relationships and combined effects, including whether particular combinations of study hours and attendance are associated with higher exam scores.

The analysis will include data cleaning, descriptive statistics, exploratory visualisations and hypothesis testing. A Power BI dashboard will present the findings interactively.

A Linear Regression model will be developed to predict exam scores from selected factors. Regression is appropriate because the target variable, Exam_Score, is numerical. The model will be evaluated on held-out test data and compared with a simple baseline.

The findings are intended to help students, parents and educators explore patterns in the dataset. Associations will not be treated as proof of cause and effect, and data and model limitations will be documented.

## Dataset Source

- **Dataset:** Student Performance Factors
- **Platform:** Kaggle
- **Uploader username:** lainguyn123
- **Source:** https://www.kaggle.com/datasets/lainguyn123/student-performance-factors
- **Licence:** CC0: Public Domain
- **Source page reviewed:** 6 October 2026
- **Original dataset size:** 6,607 rows and 20 columns

The dataset contains student exam scores alongside study habits,
attendance, and family and school factors.

The original CSV is preserved in `data/raw/`. A separate cleaned
version is saved in `data/processed/`.

The original collection method and whether the records represent
real or synthetic students have not been verified.

## Objectives

- Inspect and clean the dataset, documenting missing values, duplicates, data types and outlier decisions.
- Summarise student characteristics and exam scores using descriptive statistics.
- Explore individual factors and their combined relationships with exam performance.
- Investigate whether combinations of study hours and attendance are associated with higher scores.
- Test three hypotheses and explain the statistical results and limitations.
- Create clear visualisations using reusable code and present key findings in Power BI.
- Build and evaluate a Linear Regression model for predicting exam scores.
- Document findings, testing, fixed and unfixed bugs, and future improvements. 



---

# Change working directory

* The notebook is stored in the `jupyter_notebooks` folder. The working directory is set to the main project folder so that the notebook can access the dataset in `data` and save charts in `visualisations` using relative paths.

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\projects\\student-performance-analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'c:\\projects\\student-performance-analysis'

## 1. Load the Dataset

Load the Student Performance Factors dataset into a pandas DataFrame and preview the first five rows to check that it has loaded correctly.

## Import Libraries

In [24]:
# Check that all listed packages import successfully
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import streamlit as st
import feature_engine
import imblearn
import sklearn
import xgboost
import ipykernel
import scipy
import statsmodels.api as sm

print("All listed packages imported successfully.")

All listed packages imported successfully.


In [25]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/StudentPerformanceFactors.csv")

# Preview the first five rows
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


### Dataset Size

Check the number of rows and columns to understand the size of the dataset.

---

In [26]:
# Display the number of rows and columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 6607
Columns: 20


### Findings

The dataset contains 6,607 rows and 20 columns. Each row represents a student, and the columns describe factors relating to their performance, including their exam score.

### Data Types and Missing Values

Inspect the columns, their data types and non-null counts to identify potential missing values and check how the data is stored.

In [27]:
# Inspect column names, non-null counts and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null   int64 
 6   Previous_Scores             6607 non-null   int64 
 7   Motivation_Level            6607 non-null   object
 8   Internet_Access             6607 non-null   object
 9   Tutoring_Sessions           6607 non-null   int64 
 10  Family_Income               6607 non-null   object
 11  Teacher_Quality             6529 non-null   object
 12  School_Type                 6607 non-null   object
 13  Peer_Influence              6607 non-null   obje

### Findings

- The dataset contains 7 integer columns and 13 object columns containing categorical text.
- Three columns have missing values:
  - Teacher_Quality: 78 missing values.
  - Parental_Education_Level: 90 missing values.
  - Distance_from_Home: 67 missing values.
- The remaining 17 columns have no missing values.
- These missing values will need to be addressed during data cleaning.

### Missing Values Before Treatment

| Column | Missing values | Percentage |
|---|---:|---:|
| Teacher_Quality | 78 | 1.18% |
| Parental_Education_Level | 90 | 1.36% |
| Distance_from_Home | 67 | 1.01% |

All other columns had 0 missing values (0%). Percentages are calculated from 6,607 rows and rounded to two decimal places.

Missing values were replaced with the fixed label "Unknown". This approach does not learn any values from the data. If learned imputation is used later for machine learning, it will be fitted on the training data only.

In [ ]:
# Select numerical columns
numeric_data = df.select_dtypes(include="number")

# Calculate quartiles and the interquartile range
# This part of code is created by ChatGPt and adjusted and tested by the user.
q1 = numeric_data.quantile(0.25)
q3 = numeric_data.quantile(0.75)
iqr = q3 - q1

# Flag values outside the IQR boundaries
outliers = (
    (numeric_data < q1 - 1.5 * iqr)
    | (numeric_data > q3 + 1.5 * iqr)
)

# Count potential outliers in each column
outliers.sum()

Hours_Studied         43
Attendance             0
Sleep_Hours            0
Previous_Scores        0
Tutoring_Sessions    430
Physical_Activity      0
Exam_Score           104
dtype: int64

### Check for Duplicate Rows

Check whether any complete rows are repeated in the dataset.

In [28]:
# Count duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


### Findings

No duplicate rows were found in the dataset, so no rows need to be removed as duplicates.
No exact duplicate rows were found. Matching values could represent different students because the dataset has no unique student identifier, so identical rows would require investigation before removal. No rows were removed, leaving 6,607 rows.

### Inspect Categorical Values

Review the values and their frequencies in the three columns with missing data to inform the cleaning approach.

In [29]:
# Show category counts, including missing values
# Taken from Code Institute LMS and adapted for this project with ChatGPT assistance.
for column in ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


Teacher_Quality
Teacher_Quality
Medium    3925
High      1947
Low        657
NaN         78
Name: count, dtype: int64

Parental_Education_Level
Parental_Education_Level
High School     3223
College         1989
Postgraduate    1305
NaN               90
Name: count, dtype: int64

Distance_from_Home
Distance_from_Home
Near        3884
Moderate    1998
Far          658
NaN           67
Name: count, dtype: int64


### Findings

- The most common teacher quality category is Medium (3,925 students).
- The most common parental education level is High School (3,223 students).
- Most students live Near the school (3,884 students).
- Missing values affect fewer than 1.4% of entries in each of these columns.

Missing values will be labelled "Unknown" to retain all students without assuming their teacher quality, parental education or distance from home.

In [30]:
# Replace missing categorical values with "Unknown"
columns_to_fill = [
    "Teacher_Quality",
    "Parental_Education_Level",
    "Distance_from_Home"
]

df[columns_to_fill] = df[columns_to_fill].fillna("Unknown")

# Check the total number of remaining missing values
print("Remaining missing values:", df.isna().sum().sum())

Remaining missing values: 0


### Findings

All missing values in Teacher_Quality, Parental_Education_Level and Distance_from_Home have been replaced with "Unknown". No missing values remain, and all 6,607 rows have been retained.

### Inspect Numerical Ranges

Review summary statistics for the numerical columns to identify unusual values and potential data quality issues.

In [31]:
# Display summary statistics for numerical columns
df.describe()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


### Findings

- Hours_Studied ranges from 1 to 44, with an average of approximately 20.
- Attendance ranges from 60% to 100%, averaging approximately 80%.
- Sleep_Hours ranges from 4 to 10, averaging approximately 7.
- Previous_Scores ranges from 50 to 100.
- Exam_Score ranges from 55 to 101, with an average of approximately 67.24.
- The exam score of 101 may be invalid if the maximum permitted score is 100. Records above 100 will be inspected before deciding how to handle them.

In [32]:
# Inspect records with exam scores above 100
df.loc[df["Exam_Score"] > 100]

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
1525,27,98,Low,Medium,Yes,6,93,Low,No,5,High,High,Public,Positive,3,No,High School,Moderate,Female,101


### Findings

One record (index 1525) has an Exam_Score of 101. This is a potential data quality issue if exam scores are limited to 100. The value is retained pending confirmation of the scoring scale, as there is insufficient evidence to justify changing or removing it.

### Check for Potential Outliers

Use the interquartile range (IQR) method to identify unusually low or high numerical values. Values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR are flagged as potential outliers.

Flagged values are not automatically errors and will not be removed without further investigation.

In [33]:
# Select numerical columns
numeric_data = df.select_dtypes(include="number")

# Calculate quartiles and the interquartile range
# This part of code is created by ChatGPt and adjusted and tested by the user.
q1 = numeric_data.quantile(0.25)
q3 = numeric_data.quantile(0.75)
iqr = q3 - q1

# Flag values outside the IQR boundaries
outliers = (
    (numeric_data < q1 - 1.5 * iqr)
    | (numeric_data > q3 + 1.5 * iqr)
)

# Count potential outliers in each column
outliers.sum()

Hours_Studied         43
Attendance             0
Sleep_Hours            0
Previous_Scores        0
Tutoring_Sessions    430
Physical_Activity      0
Exam_Score           104
dtype: int64

### Findings

The IQR method identified potential outliers in three numerical columns:

| Column | Potential outliers | Percentage of students |
|---|---:|---:|
| Hours_Studied | 43 | 0.65% |
| Tutoring_Sessions | 430 | 6.51% |
| Exam_Score | 104 | 1.57% |

No potential outliers were detected in Attendance, Sleep_Hours, Previous_Scores or Physical_Activity.

Tutoring_Sessions has the most flagged values. Since most students attend between 1 and 2 sessions, the IQR method flags students attending 4 or more sessions. These values may represent valid differences in the amount of tutoring received.

**Decision:** Retain the flagged values at this stage. Statistical outliers are not necessarily errors, and removing them could exclude meaningful variation in student behaviour and performance. The exam score of 101 remains a separate, unresolved data quality concern.

These counts refer to each column separately; the same student may be flagged in more than one column.

### Check Categorical Values for Consistency

Inspect the unique values in each text column to identify inconsistent labels or unwanted spaces that could split a single category into separate groups.

In [34]:
# Display unique values in each text column
# Taken from Code Institute LMS and adapted for this project with ChatGPT assistance.
for column in df.select_dtypes(include="object").columns:
    print(f"{column}: {df[column].unique().tolist()}\n")

Parental_Involvement: ['Low', 'Medium', 'High']

Access_to_Resources: ['High', 'Medium', 'Low']

Extracurricular_Activities: ['No', 'Yes']

Motivation_Level: ['Low', 'Medium', 'High']

Internet_Access: ['Yes', 'No']

Family_Income: ['Low', 'Medium', 'High']

Teacher_Quality: ['Medium', 'High', 'Low', 'Unknown']

School_Type: ['Public', 'Private']

Peer_Influence: ['Positive', 'Negative', 'Neutral']

Learning_Disabilities: ['No', 'Yes']

Parental_Education_Level: ['High School', 'College', 'Postgraduate', 'Unknown']

Distance_from_Home: ['Near', 'Moderate', 'Far', 'Unknown']

Gender: ['Male', 'Female']



### Findings

All 13 categorical columns have consistent labels. No spelling variations, inconsistent capitalisation or visible leading or trailing spaces were found.

The "Unknown" category appears only in Teacher_Quality, Parental_Education_Level and Distance_from_Home, where missing values were replaced earlier.

No further changes to the categorical labels are required.

### Save the Cleaned Dataset

Save the processed dataset separately from the original file. Missing categorical values have been replaced with "Unknown", and all rows are retained. The unresolved exam score of 101 remains documented for further review.

In [35]:
# Ensure the processed data folder exists
# This part of code is created by ChatGPt and adjusted and tested by the user.
os.makedirs("data/processed", exist_ok=True)

# Save without adding the DataFrame index as an extra column
df.to_csv("data/processed/StudentPerformanceFactors_cleaned.csv", index=False)

print(f"Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns.")

Saved cleaned dataset: 6607 rows, 20 columns.


# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

## Conclusions and Next Steps

### Conclusions

- The dataset was inspected and all 6,607 rows were retained.
- No exact duplicate rows were found.
- Missing values in Teacher_Quality, Parental_Education_Level,
  and Distance_from_Home were replaced with `Unknown`.
- The cleaned dataset contains no remaining missing values.
- Statistical outliers were investigated and retained because
  unusual values are not necessarily errors.
- One Exam_Score value of 101 remains an unresolved data quality
  issue because the permitted scoring range has not been confirmed.
- The cleaned dataset was saved separately, preserving the original data.

### Next Steps

- Load the cleaned dataset into `Student_Performance_EDA.ipynb`.
- Review distinct values and account for the `Unknown` category
  when comparing groups.
- Explore distributions and relationships through summary
  statistics and visualisations.
- Address the project's research questions and document findings
  and limitations.